In [1]:
from lcpy.calculators.bw_int import mpLCAer
import os
from lcpy.hvs.hvs import store_scenario_results
from lcpy.calculators.env_calc import fast_calculator
import pandas as pd
from lcpy.hvs.map_dicts import create_mapping, create_list_with_unique_activities
import numpy as np

The initial configuration remains the same with the simple LCA example

In [2]:
target_dir = "path_of_directory_where_to_store_results"
os.makedirs(target_dir, exist_ok=True)

In [3]:
brightway_configuration_dictionary = {
    "path_to_brightway_project": "path_to_folder_containing_the_bw_environment_and_packages_installed_there",
    "bw_project": "bw_project_name",
    "bw_database": "bw_project_database_name",
    "bw_biosphere": "bw_project_biosphere_database_name",
    "bw_ecoinvent": "ecoinvent_database_used_name"
}

In [4]:
methods_list = [
('TRACI v2.1', 'acidification', 'acidification potential (AP)'),
('TRACI v2.1', 'climate change', 'global warming potential (GWP100)'),
('TRACI v2.1', 'ecotoxicity: freshwater', 'ecotoxicity: freshwater'),
('TRACI v2.1', 'eutrophication', 'eutrophication potential'),
('TRACI v2.1', 'human toxicity: carcinogenic', 'human toxicity: carcinogenic'),
('TRACI v2.1', 'human toxicity: non-carcinogenic', 'human toxicity: non-carcinogenic'),
('TRACI v2.1', 'ozone depletion', 'ozone depletion potential (ODP)'),
('TRACI v2.1', 'particulate matter formation', 'particulate matter formation potential (PMFP)'),
('TRACI v2.1', 'photochemical oxidant formation', 'maximum incremental reactivity (MIR)'),
]

method_units_list = ['kg SO2-Eq',
 'kg CO2-Eq',
 'CTUe',
 'kg N-Eq',
 'CTUh',
 'CTUh',
 'kg CFC-11-Eq',
 'kg PM2.5-Eq',
 'kg O3-Eq',
]

In [5]:
methods_gp = methods_list[:]
impact_categories_names = ['AP', 'GWP100', 'ECFW', 'EP', 'HTC', 'HTNC', 'ODP', 'PMFP', 'MIR']

In contrast to the simple LCA we now set the number of scenarios to a high number that basically represents the MC loops

In [6]:
scenarios = 100
timeframe = 1 #operational lifetime after construction
time_step = 1
construction_years = 0
number_of_infrastructure_processes = 0

The parametric model should now be set to include uncertainty for the MC loops. This is done below, by setting a number of parameters to raneg with some probability distributions

In [7]:
keys_nuclear_power_generation = {
    'PWR': 'bw_key_pointing_to_relevant_process',
    'BWR': 'bw_key_pointing_to_relevant_process'
}

keys_fossil_power_generation = {
    'NG_ccpp': 'bw_key_pointing_to_relevant_process',
    'NG_convpp': 'bw_key_pointing_to_relevant_process',
    'NG_cogen_conv': 'bw_key_pointing_to_relevant_process',
    'NG_cogen_cc': 'bw_key_pointing_to_relevant_process',
}

keys_res_power_generation = {
    'Hydro': 'bw_key_pointing_to_relevant_process',
    'DGE': 'bw_key_pointing_to_relevant_process',
    'Wind': 'bw_key_pointing_to_relevant_process',
}

keys_total_power_generation = {
    'Nuclear': '',
    'Fossil': '',
    'RES': '',
}

In [8]:
key_list_sub_processes = [keys_nuclear_power_generation, keys_fossil_power_generation, keys_res_power_generation]
mapping_names = create_mapping(keys_total_power_generation, key_list_sub_processes)
unique_activities = create_list_with_unique_activities(key_list_sub_processes)
my_lca = mpLCAer(4, methods_gp, brightway_configuration_dictionary)
my_lca.import_isolated_environment()
my_lca.lca_calculations(mapping_names)

In [9]:
def model(BWR_capacity, PWR_capacity, NG_ccpp_capacity, NG_cogen_conv_capacity, NG_convpp_capacity, NG_cogen_cc_capacity,
     Hydro_pumped_capacity, DGE_capacity, Wind_1_3_capacity):

    nuclear_capacity = BWR_capacity + PWR_capacity
    ng_capacity = NG_ccpp_capacity + NG_convpp_capacity + NG_cogen_conv_capacity + NG_cogen_cc_capacity
    res_capacity = Hydro_pumped_capacity + DGE_capacity + Wind_1_3_capacity
    total_capacity = nuclear_capacity + ng_capacity + res_capacity

    nuclear_exchanges_amounts = [
        PWR_capacity / nuclear_capacity,
        BWR_capacity / nuclear_capacity
    ]

    fossil_exchanges_amounts = [
        NG_ccpp_capacity / ng_capacity,
        NG_convpp_capacity / ng_capacity,
        NG_cogen_conv_capacity / ng_capacity,
        NG_cogen_cc_capacity / ng_capacity,
    ]

    res_exchanges_amounts = [
        Hydro_pumped_capacity / res_capacity,
        DGE_capacity / res_capacity,
        Wind_1_3_capacity / res_capacity,
    ]

    mp_exchanges_amounts = [
        nuclear_capacity / total_capacity,
        ng_capacity / total_capacity,
        res_capacity / total_capacity,
    ]
    exchanges_list_sp = [nuclear_exchanges_amounts, fossil_exchanges_amounts, res_exchanges_amounts]
    mapping_exchanges = create_mapping(keys_total_power_generation, exchanges_list_sp)
    my_calculator = fast_calculator()

    my_calculator.calculation_static_lcia(mapping_exchanges, my_lca.unit_impacts)
    my_calculator.calculation_impact_senarios(mp_exchanges_amounts, my_calculator.impact, 'Electricity production')

    gsa_result = my_calculator.total_impact['Electricity production'][:,:,0].T

    return gsa_result

In [10]:
from pymoo.core.problem import ElementwiseProblem, Problem
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize

In [11]:
low_bounds = np.array([1200, 700, 350, 600, 560, 230, 320, 55, 900])
high_bounds = np.array([1800, 900, 450, 800, 700, 280, 579, 115, 1100])

In [12]:
class Single(ElementwiseProblem):
    def __init__(self):
        super().__init__(
            n_var=9,
            n_obj=1,
            xl = low_bounds,
            xu = high_bounds,
            elementwise_evaluation=True
        )

    def _evaluate(self, x, out, *args, **kwargs):

        BWR_capacity, PWR_capacity, NG_ccpp_capacity, NG_cogen_conv_capacity, NG_convpp_capacity, NG_cogen_cc_capacity, Hydro_pumped_capacity, DGE_capacity, Wind_1_3_capacity = [np.array([value]).reshape((1,1)) for value in x]

        F = model(BWR_capacity, PWR_capacity, NG_ccpp_capacity, NG_cogen_conv_capacity, NG_convpp_capacity, NG_cogen_cc_capacity, Hydro_pumped_capacity, DGE_capacity, Wind_1_3_capacity)
        # If your model returns shape (1,1) for a single objective:
        out["F"] = F[0, 1]

In [13]:
problem = Single()
algorithm = GA(pop_size=100)
res = minimize(
    problem,
    algorithm,
    ('n_gen', 200),
    seed=42,
    verbose=True
)

n_gen  |  n_eval  |     f_avg     |     f_min    
     1 |      100 |  0.2067939400 |  0.1869344557
     2 |      200 |  0.1977290292 |  0.1846418791
     3 |      300 |  0.1914133956 |  0.1830899344
     4 |      400 |  0.1872014309 |  0.1810320115
     5 |      500 |  0.1844001641 |  0.1777006858
     6 |      600 |  0.1821322401 |  0.1777006858
     7 |      700 |  0.1801027594 |  0.1758593388
     8 |      800 |  0.1785020054 |  0.1752974707
     9 |      900 |  0.1772347551 |  0.1749962673
    10 |     1000 |  0.1762384803 |  0.1748142456
    11 |     1100 |  0.1754428480 |  0.1732529561
    12 |     1200 |  0.1748947696 |  0.1732529561
    13 |     1300 |  0.1744246893 |  0.1732529561
    14 |     1400 |  0.1739793147 |  0.1732529561
    15 |     1500 |  0.1736014914 |  0.1726361395
    16 |     1600 |  0.1733021163 |  0.1726361395
    17 |     1700 |  0.1730698415 |  0.1725154221
    18 |     1800 |  0.1728295674 |  0.1724334948
    19 |     1900 |  0.1726445003 |  0.1723857852


In [14]:
class Multi(ElementwiseProblem):
    def __init__(self):
        super().__init__(
            n_var=9,
            n_obj=2,
            n_constr=0,
            xl=low_bounds,
            xu=high_bounds,
            elementwise_evaluation=True
        )

    def _evaluate(self, x, out, *args, **kwargs):

        BWR_capacity, PWR_capacity, NG_ccpp_capacity, NG_cogen_conv_capacity, NG_convpp_capacity, NG_cogen_cc_capacity, Hydro_pumped_capacity, DGE_capacity, Wind_1_3_capacity = [np.array([value]).reshape((1,1)) for value in x]

        F = model(BWR_capacity, PWR_capacity, NG_ccpp_capacity, NG_cogen_conv_capacity, NG_convpp_capacity, NG_cogen_cc_capacity, Hydro_pumped_capacity, DGE_capacity, Wind_1_3_capacity)

        out["F"] = F[0, :2]

In [15]:
mo_problem = Multi()
mo_algo    = NSGA2(pop_size=100)
mo_res     = minimize(
    mo_problem,
    mo_algo,
    ('n_gen', 200),
    seed=42,
    verbose=True
)

n_gen  |  n_eval  | n_nds  |      eps      |   indicator  
     1 |      100 |      1 |             - |             -
     2 |      200 |      2 |  1.0000000000 |         ideal
     3 |      300 |      3 |  0.7953698026 |         ideal
     4 |      400 |      1 |  0.0039195857 |         nadir
     5 |      500 |      2 |  1.5994621335 |         ideal
     6 |      600 |      1 |  0.0026447153 |         nadir
     7 |      700 |      2 |  2.5132973832 |         ideal
     8 |      800 |      1 |  0.0029477804 |         nadir
     9 |      900 |      3 |  1.0000000000 |         ideal
    10 |     1000 |      5 |  0.6084538988 |         ideal
    11 |     1100 |      2 |  4.9062560224 |         ideal
    12 |     1200 |      2 |  1.1404464726 |         ideal
    13 |     1300 |      2 |  0.000000E+00 |             f
    14 |     1400 |      4 |  0.2650169118 |         ideal
    15 |     1500 |      2 |  1.7392040782 |         ideal
    16 |     1600 |      3 |  1.1117929068 |         ide

# Contributions:

- Show how to do it for the modular approach
- make it for emissions of an environmental flow
- change sampling strategies to allow for more probability distributions